# Setup and prompting

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.core.pylabtools import figsize

import seaborn as sns

import numpy as np
import pandas as pd

In [2]:
from transformers import pipeline
from datasets import load_dataset


ds = load_dataset("openai/gsm8k", "main")
pipe = pipeline("text-generation", model="NousResearch/Llama-3.2-1B",
                do_sample=True, temperature=1., return_full_text=False)
    

import error: No module named 'triton'


Device set to use mps:0


In [3]:
q1 = ds['train'][0]['question']
print(q1)

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?


In [4]:
pipe(q1)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': ' 11\nSarah, Jane, and Katie sold 12, 4, and 6 cookies, respectively. Determine a way to determine the total number of cookies that all three girls sold. 15\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? 11\nSarah, Jane, and Katie sold 12, 4, and 6 cookies, respectively. Determine a way to determine the total number of cookies that all three girls sold. 15\nSarahs, Janes and Katyss cookies sold together in the total of 34 cookies.\nThe cookies were sold equally from one batch after the other.\nIn one batch Sarah sold six cookies, one batch later she sold two cookies, finally she sold two cookies.\nWhat is the total number of cookies sold altogether?\nStep 2: Use the information from step 1\n  1. Sarah sold five more cookies that morning, meaning total cookies = 24.\n  2. 24/3 = 8 or 8 cookies were sold altogether.\nAnswer: B\nExplanation: Sarah, Jane, and Ka

## Formatting prompts to get answers out

Simple zero-shot COT prompting

In [5]:
prompt_format = "Q: {question}\nA: Let's think step-by-step:"

In [6]:
print(prompt_format.format(question=q1))

Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
A: Let's think step-by-step:


In [7]:
pipe( prompt_format.format(question=q1))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': ' April = 48; April and May = 50 clips * 2 = 100; April'}]

In [8]:
pipe( prompt_format.format(question=q1))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': '........................\nQ: After Natalie sold 48 clips in April, how many clips did she sell'}]

### llama specific formatting

In [9]:
prompt_template = """<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST]"""

In [10]:
pipe(prompt_template.format(system_prompt="""You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information.
""", user_message='What is 2+2?'))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': ''}]

This doesn't seem to show much promise without further optimization

## Instruction Tuned version

In [11]:
model = pipeline("text-generation", model="unsloth/Llama-3.2-1B-Instruct",
                do_sample=True, temperature=1.)


Device set to use mps:0


Zero-shot CoT prompting with formatting instructions

In [12]:
prompt_format = """Please answer the following question, with your final answer stated at the end as 'Final answer: ' \nQ: {question}\nA: Let's think step-by-step:"""

In [13]:
prompt_format.format(question=q1)

"Please answer the following question, with your final answer stated at the end as 'Final answer: ' \nQ: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nA: Let's think step-by-step:"

In [14]:
output = model( prompt_format.format(question=q1))[0]['generated_text']

In [15]:
print(output)

Please answer the following question, with your final answer stated at the end as 'Final answer: ' 
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
A: Let's think step-by-step: 
Natalia sold a total of 48 + 48 = 96 clips in April and May.
To find half of 48, we need to divide 48 by 2.
48 ÷ 2 = 24
So, Natalia sold 24 clips in May, and a total of 96 clips in April and May. 
Final answer: 96.


In [16]:
output.split('Final answer: ')

["Please answer the following question, with your final answer stated at the end as '",
 "' \nQ: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nA: Let's think step-by-step: \nNatalia sold a total of 48 + 48 = 96 clips in April and May.\nTo find half of 48, we need to divide 48 by 2.\n48 ÷ 2 = 24\nSo, Natalia sold 24 clips in May, and a total of 96 clips in April and May. \n",
 '96.']

Yes. Starting from the instruction-tuned version is super helpful.

### We need a collection of prompts to try out.

In [17]:
prompt_format_2 = """Please answer the following question, with your final answer stated at the end as 'Final answer: ' \nQ: {question}\nAns: """

In [18]:
print(model( prompt_format_2.format(question=q1))[0]['generated_text'])

Please answer the following question, with your final answer stated at the end as 'Final answer: ' 
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Ans: 192
Q: A bookshelf has 5 shelves. If 3 books are placed on each shelf, how many books will be placed on the bookshelf? 
Ans: 15
Q: A rectangular garden measures 12 meters in length and 8 meters in width. What is the area of the garden? 
Ans: 96
Q: If it takes 5 people 5 days to clean a job, then how many people will it take to clean the job in 2 days? 
Ans: 1

Final answer: None of the above.


In [19]:
prompt_format_3 = """Please answer the following question. State your reasoning between two tags <reasoning> and </reasoning> and you final answer at the end between tags <answer> and </answer> \nQuestion: {question}\nAnswer: """

In [20]:
print(model( prompt_format_3.format(question=q1))[0]['generated_text'])

Please answer the following question. State your reasoning between two tags <reasoning> and </reasoning> and you final answer at the end between tags <answer> and </answer> 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer: 96

<reasoning>
In April, Natalia sold 48 clips to 48 friends. Then she sold half of 48, which is 24, so she sold 48+24=72 clips in April. Then she sold half of 72, which is 36, so she sold 72+36=108 clips in May.
</reasoning>

<reasoning>
Natalia sold 48 clips in April. In May, she sold half of 48, which is 24. The total number of clips sold in both months is the sum of the number of clips sold in each month, which is 48+24 = 72. However, the problem states that Natalia sold half as many clips in May as in April, so it is not possible to simply add 72 and 48 to find the answer.
</reasoning>

<reasoning>
The information given in the problem can

In a couple of tries, this one seems to work less well. It often produces the correct answer, but the formatting is sensitive to the details of the prompt and to sampling

In [21]:
prompt_format_4 = """Please answer the following question. State your final answer at the end between tags <answer> and </answer> \nQuestion: {question} """

In [22]:
print(model( prompt_format_4.format(question=q1))[0]['generated_text'])

Please answer the following question. State your final answer at the end between tags <answer> and </answer> 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  a) 104 
b) 120
c) 128
d) 144

Answer: c) 128
Explanation: First, find how many clips Natalia sold in April. Since 1 answer is equal to the number of people, multiply 48 by 2: 48 x 2 = 96 clips in April. Next, since she sold half as many clips in May, multiply 96 by 0.5. 
96 x 0.5 = 48 clips in May.
48 clips in April + 48 clips in May = 96 + 48 = 144 clips.
The correct answer is d) 144.


Interesting to see that this one generates more questions in the same format. I would guess that the model has been trained on the benchmark or on other datasets that heavily resemble the benchmark.

In [23]:
prompt_format_5 = """Please answer the following question, reasoning step by step and stating the final answer at the end. \nQuestion: {question} """

In [24]:
answer = model( prompt_format_5.format(question=q1))[0]['generated_text']
print(answer)

Please answer the following question, reasoning step by step and stating the final answer at the end. 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  To determine the answer, we can use a Venn diagram and basic addition.
Step 1: Identify the number of friends Natalia had in April.
Step 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.
Step 3: Calculate the total number of clips sold in April and May using a Venn diagram.
Step 4: Add the number of clips sold in April and May to find the total.

Let's break down the steps and calculate the total number of clips sold.

Step 1: Identify the number of friends Natalia had in April.
Natalia sold clips to 48 of her friends in April.

Step 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.
Natalia sold half as many clips in May, so the numbe

This prompt seems to work quite well, and consistently (over a couple of samples) produces well formatted answers.

In [25]:
print(answer[:-12])

Please answer the following question, reasoning step by step and stating the final answer at the end. 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  To determine the answer, we can use a Venn diagram and basic addition.
Step 1: Identify the number of friends Natalia had in April.
Step 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.
Step 3: Calculate the total number of clips sold in April and May using a Venn diagram.
Step 4: Add the number of clips sold in April and May to find the total.

Let's break down the steps and calculate the total number of clips sold.

Step 1: Identify the number of friends Natalia had in April.
Natalia sold clips to 48 of her friends in April.

Step 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.
Natalia sold half as many clips in May, so the numbe

In [26]:
pipe(answer[:-12])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': '144$ clips.'}]

In [27]:
print(answer[:-34])

Please answer the following question, reasoning step by step and stating the final answer at the end. 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  To determine the answer, we can use a Venn diagram and basic addition.
Step 1: Identify the number of friends Natalia had in April.
Step 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.
Step 3: Calculate the total number of clips sold in April and May using a Venn diagram.
Step 4: Add the number of clips sold in April and May to find the total.

Let's break down the steps and calculate the total number of clips sold.

Step 1: Identify the number of friends Natalia had in April.
Natalia sold clips to 48 of her friends in April.

Step 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.
Natalia sold half as many clips in May, so the numbe

In [28]:
pipe(answer[:-34])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': 'In the end, Natalia sold 144 clips to 48 friends in April and then 72'}]

The model that is not instruction tuned seems to want to generate multiple choice responses

In [29]:
model(answer[:-12])

[{'generated_text': "Please answer the following question, reasoning step by step and stating the final answer at the end. \nQuestion: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  To determine the answer, we can use a Venn diagram and basic addition.\nStep 1: Identify the number of friends Natalia had in April.\nStep 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.\nStep 3: Calculate the total number of clips sold in April and May using a Venn diagram.\nStep 4: Add the number of clips sold in April and May to find the total.\n\nLet's break down the steps and calculate the total number of clips sold.\n\nStep 1: Identify the number of friends Natalia had in April.\nNatalia sold clips to 48 of her friends in April.\n\nStep 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.\nNatalia sold half 

Instruction tuned is on it.

In [30]:
model(answer[:-34])

[{'generated_text': "Please answer the following question, reasoning step by step and stating the final answer at the end. \nQuestion: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  To determine the answer, we can use a Venn diagram and basic addition.\nStep 1: Identify the number of friends Natalia had in April.\nStep 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.\nStep 3: Calculate the total number of clips sold in April and May using a Venn diagram.\nStep 4: Add the number of clips sold in April and May to find the total.\n\nLet's break down the steps and calculate the total number of clips sold.\n\nStep 1: Identify the number of friends Natalia had in April.\nNatalia sold clips to 48 of her friends in April.\n\nStep 2: Determine the number of clips sold in May by dividing the number of friends in April by 2.\nNatalia sold half 

Less success with a more open ended prompt.

## Real chain of thought prompting

In [31]:
dialog = """Quesion: {question} \n Answer: {answer} \n Question: {test_question} """ 

In [32]:
cot = dialog.format(**ds['train'][1], test_question=ds['train'][0]['question'])
print(cot)

Quesion: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? 
 Answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10 
 Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? 


In [33]:
print(model(cot)[0]['generated_text'])

Quesion: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? 
 Answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10 
 Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  A) 150 B) 160 C) 180 D) 200 
 Answer: Natalia sold 48 clips in April and half of those in May, which is 48 / 2 = 24 clips in May. In total, she sold 48 + 24 = 72 clips. The correct answer is  D) 200.


In [34]:
print(pipe(cot)[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


10
Answer: To find the total number of clips Natalia sold, we can use the formula


Also interesting to see this. It almost seems to indicate that chain of thought style prompting was used to construct the insturction tuned network.

In [35]:
qa_pair = """Quesion: {question}\nAnswer: {answer}\n"""
last_question = """Question: {test_question} """ 
cot_prompt = """"""

for i in range(1, 4):
    cot_prompt += qa_pair.format(**ds['train'][i])

cot_prompt += last_question.format(test_question=ds['train'][0]['question'])


In [36]:
print(cot_prompt)

Quesion: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
Quesion: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Answer: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5
Quesion: Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?
Answer: Maila read 12 x 2 = <<12*2=24>>24 pages tod

In [37]:
print( model(cot_prompt)[0]['generated_text'] )

Quesion: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
Quesion: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Answer: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5
Quesion: Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?
Answer: Maila read 12 x 2 = <<12*2=24>>24 pages tod

In [38]:
print( pipe(cot_prompt)[0]['generated_text'] )

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


1.6, 5.6, 7, 8
2. The area of


Chain of thought prompting is interesting. It does create a uniformity of answer formats, but at the same time causes the model to output additional practice arithmetic problems

In [67]:
prompt_format_6 = """Please answer the following question, reasoning step by step and stating the final answer at the end as 'The final answer is: __'. \nQuestion: {question}\n"""

In [68]:
print(prompt_format_6)

Please answer the following question, reasoning step by step and stating the final answer at the end as 'The final answer is: __'. 
Question: {question}



In [69]:
answer = model( prompt_format_6.format(question=q1))[0]['generated_text']
print(answer)

Please answer the following question, reasoning step by step and stating the final answer at the end as 'The final answer is: __'. 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Step 1: Determine the number of clips sold in April.
Natalia sold 48 clips in April.

Step 2: Determine the number of clips sold in May.
Natalia sold half as many clips in May, which is (48/2) = 24 clips.

Step 3: Add the number of clips sold in April and May.
To find the total number of clips sold, we add the number of clips sold in April (48) to the number of clips sold in May (24).
48 + 24 = 72

The final answer is: 72.


In [63]:
print(prompt_format_5)

Please answer the following question, reasoning step by step and stating the final answer at the end. 
Question: {question} 


In [70]:
answer = model( prompt_format_5.format(question=q1))[0]['generated_text']
print(answer)

Please answer the following question, reasoning step by step and stating the final answer at the end. 
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?  If each clip was sold for 5 dollars, how much money did Natalia make altogether?

## Step 1: Calculate the number of clips sold in April.
Natalia sold clips to 48 of her friends in April.

## Step 2: Calculate the number of clips sold in May.
Natalia sold half as many clips in May as she did in April. Since she sold 48 clips in April, she sold 48 / 2 = 24 clips in May.

## Step 3: Calculate the total number of clips sold in April and May.
To find the total number of clips sold, we need to add the number of clips sold in April and May. Total clips sold = 48 (April) + 24 (May) = 72.

## Step 4: Calculate the total amount of money made altogether.
Each clip was sold for 5 dollars. To find the total amount of money made alto

In [66]:
prompt_format_6

"Please answer the following question, reasoning step by step and stating the final answer at the end as 'The final answer is: __'. \nQuestion: {question}"

## Key takeaways

Instruction tuned models are certainly the place to start.

Things to try:
1. Prompts 5 and 6 are simple and seem to produce good results
2. COT prompting produces very nice formatting
3. Different levels of temperature

Random idea: Different search strategies?

In [98]:
model(prompt_format_5.format(question=q1), num_return_sequences=8)

[{'generated_text': 'Please answer the following question, reasoning step by step and stating the final answer at the end. \nQuestion: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? 40 or more, please use specific numbers for the answers.\na) 112 clips\nb) 120 clips\nc) 130 clips\nd) 126 clips\n\nStep 1: Calculate the total number of clips sold in April.\nNatalia sold 48 clips to her friends in April.\n\nStep 2: Calculate the total number of clips sold in May.\nShe sold half as many clips in May, which is 48 / 2 = 24 clips.\n\nStep 3: Calculate the total number of clips sold in both April and May.\nAdd the number of clips sold in April and May: 48 + 24 = 72 clips.\n\nHowever, the question asks for the total number of clips sold in April and May, and the answer choice is 40 or more. The total number of clips sold is 72 clips, which is less than 40 clips. Therefore, none of the an

# Part 2: running into speed problems with inference

In [1]:
from transformers import pipeline
from datasets import load_dataset


import error: No module named 'triton'


Could this lack of triton be the issue? It seems to be linux only, so for cloud services

In [3]:
model = pipeline("text-generation", model="unsloth/Llama-3.2-1B-Instruct",
                 do_sample=True, temperature=1., max_new_tokens=1024)

Device set to use mps:0


In [4]:
dataset_base = load_dataset("openai/gsm8k", "main")
prompt = "Please answer the following question, reasoning step by step and stating the final answer at the end.\nQuestion: {question}\n"
def process_datapoint(datapoint):
    answer = datapoint.pop('answer').split('####')[-1].strip()
    question = prompt.format(**datapoint) 
    return {'question': question, 'gt': answer}

dataset_processed = dataset_base['train'].map(process_datapoint)

In [8]:
%%time
model(dataset_processed[0:10]['question'])

CPU times: user 3min 29s, sys: 17.3 s, total: 3min 46s
Wall time: 3min 51s


[[{'generated_text': 'Please answer the following question, reasoning step by step and stating the final answer at the end.\nQuestion: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nStep 1: Calculate the total number of clips sold in April and May.\nStep 2: Since Natalia sold half as many clips in May compared to April, the number of clips sold in May is half of the number sold in April.\nStep 3: We will use the steps above to calculate the number of clips sold.\nStep 1: Calculate the total number of clips sold in April and May.\nStep 2: Since Natalia sold half as many clips in May compared to April, the number of clips sold in May is half of the number sold in April.\nStep 3: We will use the steps above to calculate the number of clips sold.\nStep 1: Calculate the total number of clips sold in April and May.\nStep 2: Since Natalia sold half as many clips in May compared to Apr

In [ ]:
%%time
model(dataset_processed[0:10]['question'])

## Looking at the outputs again

In [1]:
from transformers import pipeline
model = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct",
    device_map='auto', do_sample=True, temperature=1., max_new_tokens=400
)
model.tokenizer.pad_token_id = model.model.config.eos_token_id[2]
_ = model.model.eval()

import error: No module named 'triton'


Device set to use mps


In [2]:
from datasets import load_dataset
questions = load_dataset("openai/gsm8k", "main")

prompt = "Please solve the following math problem, reasoning step by step and stating the final answer at the end.\nQuestion: {question}\n"

def process_datapoint(datapoint):
    answer = datapoint.pop('answer').split('####')[-1].strip()
    question = prompt.format(**datapoint) 
    return {'question': question, 'gt': answer}

questions = questions['train'].map(process_datapoint)

In [3]:
print(prompt)

Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: {question}



In [4]:
i = 10
batch_size=32
batch = questions[i*batch_size:(i+1)*batch_size]

In [5]:
outs = model(batch['question'], batch_size=batch_size, num_return_sequences=8)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [6]:
results = list(zip(outs, batch['gt']))

In [7]:
results[6][1]

'37'

In [8]:
print(results[6][0][0]['generated_text'])

Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: Fred had 236 dollars to spend on 6 books. After buying them, he had 14 dollars. On average, how much did each book cost?
 proceeding with the steps,
he spent $x on the first book. The remaining $236 - $x cost for the 5 other books. So, he spent a total of $x + 5(236 - x) dollars.

This expression simplifies to $x + 1150 - 5x.
Combine like terms to get -4x + 1150 dollars spent in total.
Fred then spent $14, so the equation for the situation is:
\frac{-4x + 1150}{x + 5(236 - x)} = 14

The first step is to cross multiply to solve for x. The equation becomes:
-4x + 1150 = 14(x + 5(236 - x))
Expand the expression on the right side to get:
-4x + 1150 = 14x + 3320 -14x

Next, combine the like terms to get:
-4x + 1150 = 3320
Subtract 1150 from both sides to get:
-4x = 2250

Finally, divide both sides by -4 to get:
x = 562.5

The final answer is $562.5.


There was a bug: not using the ground truth in the answer checker.

## Additional tests

### Double check the answer checker

In [9]:
from solution_checker import AnswerChecker

In [10]:
answer_checker = AnswerChecker()

Python(12214) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(12216) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [11]:
for result in results:
    answer_checker.add_result(result)

In [12]:
answer_checker.shutdown()
answer_checker.worker.join()

In [16]:
answer_checker.summarize()

{'good_formatting': 0.484375,
 'perfect_formatting': 0.421875,
 'majorityOf8': 0.28125,
 'passOf8': 0.75,
 'num_samples': 32}

In [151]:
manual_check = [ 0, 0, 5, 1, 7, 5, 4, 1, 0, 2, 6,  6, 1, 1, 0, 3, 3, 0, 5, 1, 1, 4, 0, 0, 4, 6, 4, 6, 3, 5, 1] 

In [150]:
ind = 31
results[ind][1]

'69'

In [148]:
for i in range(8):
    print( results[ind][0][i]['generated_text'])
    print('################################################')

Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: Jackson’s mom packs him a peanut butter and jelly sandwich for his school lunch on Wednesdays and Fridays. There are 36 weeks of school and Jackson has only missed 1 Wednesday and 2 Fridays. How many peanut butter and jelly sandwiches did he eat for lunch this school year?
",

## Step 1: Determine the total number of school days in a year
There are 52 weeks in a school year. However, Jackson has only missed 1 Wednesday and 2 Fridays, so the total number of school days he missed is 1 (Wednesday) + 2 (Fridays) = 3.

## Step 2: Calculate the total number of school days
To find the total number of school days Jackson attended, subtract the number of missed days from the total number of weeks. Total days = 52 - 3 = 49.

## Step 3: Calculate the total number of peanut butter and jelly sandwiches eaten
Since Jackson eats a peanut butter and jelly sandwich on Wednesdays and Friday

In [152]:
len(manual_check)

31

In [154]:
import numpy as np
sum(np.array(manual_check) > 4) / 31

np.float64(0.2903225806451613)

In [156]:
sum(np.array(manual_check) > 0) / 31

np.float64(0.7741935483870968)

Slightly different, but for the most part, consistent with the 

#### One question had the wrong answer:
Incorrect answer: index 23
```
Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: Kim's TV uses 125 watts of electricity per hour.  She runs it for 4 hours a day.  If electricity cost 14 cents per kw/h how many cents does her TV cost to run for a week?
```
claimed ans: 49


In [158]:
results[23][1]

'49'

In [166]:
0.5 *7 *14

49.0

500 wh / day -> 3.5 kwh / week -> 49 cents / week

Ok, so the answer is correct, but it is a little tricky since:
1. it relies on knowing that 1 kw = 1000 watts
2. there is a typo: should read `electricity cost 14 cents per kwh`
3. `watts of electricity per hour` is redundant, and strictly speaking, not the right units.

#### Aside:
An interesting direction could be to look for this sort of 'implicit knowledge' that, when introduced into the context helps the network solve the problem. (Or for evidence of this in the chain of though reasoning)

In [167]:
for i in range(8):
    print( results[23][0][i]['generated_text'])
    print('################################################')

Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: Kim's TV uses 125 watts of electricity per hour.  She runs it for 4 hours a day.  If electricity cost 14 cents per kw/h how many cents does her TV cost to run for a week?

################################################
Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: Kim's TV uses 125 watts of electricity per hour.  She runs it for 4 hours a day.  If electricity cost 14 cents per kw/h how many cents does her TV cost to run for a week?

################################################
Please solve the following math problem, reasoning step by step and stating the final answer at the end.
Question: Kim's TV uses 125 watts of electricity per hour.  She runs it for 4 hours a day.  If electricity cost 14 cents per kw/h how many cents does her TV cost to run for a week?
`

## Step 1: Convert watts to kilowatts
T

Despite the difficulties mentioned above, the network does get the correct answer once. This is nice for displaying the diversity of reasoning that the model can produce. Other samples do, in fact, get confused by the kwh vs kw/h typo.

Also interesting to note:
1. the first tokens output by the model are very strange
2. there are a lot of empty samples that terminate right away.

### Asking the model for feedback:
Can it find typos or other issues with the question?

In [168]:
baseline_dataset = load_dataset("openai/gsm8k", "main")

In [171]:
baseline_dataset['train'][10*32 + 23]

{'question': "Kim's TV uses 125 watts of electricity per hour.  She runs it for 4 hours a day.  If electricity cost 14 cents per kw/h how many cents does her TV cost to run for a week?",
 'answer': 'She uses 4 hours/day * 125 watts/hour=<<4*125=500>>500 watt hours/day\nThat means she uses 500 watt hours/day * 7 days/week = <<500*7=3500>>3500 watt hours/week\nThis is 3500 watt hours / 1000 watts/kw = <<3500/1000=3.5>>3.5 kw hours\nSo her TV cost 3.5 kw hours/week * $0.14/kw hour = $<<3.5*0.14=0.49>>0.49/week\n#### 49'}

In [183]:
prompt = "Are there any typos or issues that might make the question difficult to answer?\nQuestion: {question}"
chat_session = prompt.format(**baseline_dataset['train'][10*32 + 23])

messages = [
    {"role": "system", "content": "You are chatbot for teachers who want to check that their test questions make sense. Answering the questions is less important that giving good feedback."},
    {"role": "user", "content": chat_session},
]

In [185]:
question_feedback = model( messages, num_return_sequences=8)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [189]:
for i in range(8):
    print( question_feedback[i]['generated_text'][2]['content'])
    print('################################################')

To determine the cost of the TV to run for a week, we need to follow these steps:

1. Convert the power consumption from watts to kilowatts:
   125 watts ÷ 1000 = 0.125 kw/h
2. Calculate the daily energy consumption:
   0.125 kw/h x 4 hours = 0.5 kw/h/day
3. Calculate the weekly energy consumption:
   0.5 kw/h/day x 7 days = 3.5 kw/h/week
4. Calculate the weekly energy cost:
   3.5 kw/h/week x 14 cents/kw/h = $0.49/week

So, the TV costs $0.49 to run for a week.
################################################
The question is clear, and the math is straightforward.

However, here are a few potential issues:

1. The question mentions "watts" instead of "kW". To run the TV for 1 hour, you need 125 / 1000 = 0.125 kW, not 125 watts. You can fix this by converting watts to kW.

Corrected question: Kim's TV uses 1.25 kW of electricity per hour. She runs it for 4 hours a day. If electricity cost 14 cents per kW/h, how many cents does her TV cost to run for a week?
############################

The last two got the corrected question that I was looking for. Only one of them got the right answer lol. Again, very cool to see the different chaines of reasoning that appear.

In some ways, it almost feels like this is approaching an 'agentic' workflow...

In [191]:
prompt = "What is this question trying to test?\nQuestion: {question}"
chat_session = prompt.format(**baseline_dataset['train'][10*32 + 23])

messages = [
    {"role": "system", "content": "You are chatbot for school administrators who want to check their curriculum. Answering the questions is less important that giving good feedback."},
    {"role": "user", "content": chat_session},
]
question_feedback = model( messages, num_return_sequences=8)

for i in range(8):
    print( question_feedback[i]['generated_text'][2]['content'])
    print('################################################')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


This question is trying to test the student's ability to:

1. Convert units (watts to kWs and kWs to cents) for electricity consumption.
2. Perform unit conversion (kW to hour).
3. Calculate daily energy consumption (kWs/h).
4. Multiply daily energy consumption by number of days in a week to find weekly energy consumption.
5. Convert weekly energy consumption to cents (kWh * 100 cents/kWh).

However, as you mentioned, the actual question is more focused on giving good feedback rather than checking the curriculum, which implies a more detailed and contextualized explanation. The correct answer is 525 cents.
################################################
This question is trying to test the student's ability to perform mathematical calculations, specifically:

1. Converting units (watts to kw/h)
2. Calculating energy consumption over time (hours to days)
3. Converting costs to cents

The student needs to follow these steps to solve the problem:

1. Convert 125 watts to kw/h: 125 watts =

#### Sumary:
Honestly, it feels like these questions are gradually approaching the motivations for agentic workflows and reasoning models.